# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hussaintinwala2/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Age Curve

The paper reports an observed relationship between content age and performance.

**Methodology question:** How is the outcome being defined for this comparison, and are observations balanced across different content ages? I would want to check whether the observed relationship could partly reflect differences in the types of pages that are older versus newer.

### Finding 2 — The CTR Cliff

The paper reports an observed relationship between CTR and search performance, with very low CTR appearing as an important signal.

**Methodology question:** How is the CTR comparison constructed, and does the validation or comparison design support treating this as a useful predictive signal rather than a causal effect? I would check whether impressions, search position, and other search conditions are handled consistently.

These are constructive methodology questions rather than judgments about whether the findings are correct.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before/after result

The random split produced a measured ROC-AUC of **0.586**. When the same Logistic Regression approach was evaluated using a client-grouped split, the measured ROC-AUC was **0.538**.

The grouped evaluation reduced ROC-AUC by **0.048**. This indicates that part of the signal observed under the random split does not transfer as strongly when the model is tested on clients it did not see during training.

The client-grouped result is therefore the more appropriate estimate for this validation question. The model should be treated as directional decision-support evidence rather than as a reliable standalone predictor.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All
# ML-09 Section 2
# Rebuild Week-5 model and compare random vs client-grouped validation.

import os
import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# 1. Connect to the warehouse
# ---------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
}

print("DuckDB connected.")

# ---------------------------------------------------------
# 2. Build March 2026 features and April 2026 outcome
# ---------------------------------------------------------

model_data = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march

        FROM {TABLES['fact_daily']}

        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-04-01'

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    april AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions_april

        FROM {TABLES['fact_daily']}

        WHERE report_date >= DATE '2026-04-01'
          AND report_date < DATE '2026-05-01'

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.impressions_march,
        m.clicks_march,
        m.avg_position_march,
        a.impressions_april

    FROM march m

    INNER JOIN april a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id

    WHERE m.impressions_march > 0
""").df()

print(f"Modeling rows: {len(model_data):,}")
print(f"Clients: {model_data['client_hash_id'].nunique():,}")

# ---------------------------------------------------------
# 3. Create the same Week-5 target and features
# ---------------------------------------------------------

model_data["ctr_march"] = (
    model_data["clicks_march"] /
    model_data["impressions_march"].replace(0, np.nan)
)

model_data["is_declining"] = (
    model_data["impressions_april"] <
    0.8 * model_data["impressions_march"]
).astype(int)

feature_cols = [
    "impressions_march",
    "clicks_march",
    "avg_position_march"
]

model_data = model_data.dropna(
    subset=feature_cols + ["is_declining", "client_hash_id"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

print(f"Usable modeling rows: {len(model_data):,}")
print(f"Positive rate: {y.mean():.3f}")

# ---------------------------------------------------------
# 4. Define the Week-5 Logistic Regression model
# ---------------------------------------------------------

def make_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

# ---------------------------------------------------------
# 5. BEFORE: random train/test split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = make_model()
random_model.fit(X_train, y_train)

random_prob = random_model.predict_proba(X_test)[:, 1]

random_auc = roc_auc_score(
    y_test,
    random_prob
)

print("\n--- BEFORE: Random Split ---")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")
print(f"ROC-AUC:       {random_auc:.3f}")

# ---------------------------------------------------------
# 6. AFTER: client-grouped split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

grouped_model = make_model()
grouped_model.fit(
    X_train_g,
    y_train_g
)

grouped_prob = grouped_model.predict_proba(
    X_test_g
)[:, 1]

grouped_auc = roc_auc_score(
    y_test_g,
    grouped_prob
)

print("\n--- AFTER: Client-Grouped Split ---")
print(f"Training rows: {len(X_train_g):,}")
print(f"Test rows:     {len(X_test_g):,}")
print(f"Training clients: {groups_train.nunique():,}")
print(f"Test clients:     {groups_test.nunique():,}")
print(
    f"Client overlap:   "
    f"{len(set(groups_train) & set(groups_test))}"
)
print(f"ROC-AUC:           {grouped_auc:.3f}")

# ---------------------------------------------------------
# 7. Before/after comparison
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "validation_design": [
        "Random split (before)",
        "Client-grouped split (honest)"
    ],
    "roc_auc": [
        random_auc,
        grouped_auc
    ]
})

print("\n=== BEFORE / AFTER COMPARISON ===")
display(comparison)

print(
    "\nObserved change in ROC-AUC: "
    f"{grouped_auc - random_auc:+.3f}"
)

DuckDB connected.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 176,737
Clients: 47
Usable modeling rows: 176,737
Positive rate: 0.532

--- BEFORE: Random Split ---
Training rows: 132,552
Test rows:     44,185
ROC-AUC:       0.588

--- AFTER: Client-Grouped Split ---
Training rows: 133,473
Test rows:     43,264
Training clients: 35
Test clients:     12
Client overlap:   0
ROC-AUC:           0.538

=== BEFORE / AFTER COMPARISON ===


,validation_design,roc_auc
0,Random split (before),0.587634
1,Client-grouped split (honest),0.537685



Observed change in ROC-AUC: -0.050


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

The final model uses three decision-point features: `impressions_march`, `clicks_march`, and `avg_position_march`.

These features are calculated only from March 2026 search performance. The target, `is_declining`, is calculated from April 2026 impressions, so the future outcome is kept separate from the model inputs.

I also excluded identifiers such as `client_hash_id` from the predictive features. The client identifier is used only to create the grouped validation split.

**Verdict:** No observed future-window leakage in the final feature set. The features are measured before the April outcome and are therefore available at the decision point.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Leakage audit

print("=== FINAL FEATURE SET ===")

feature_audit = pd.DataFrame({
    "feature": [
        "impressions_march",
        "clicks_march",
        "avg_position_march"
    ],
    "source_window": [
        "March 2026",
        "March 2026",
        "March 2026"
    ],
    "used_for_target": [
        "No",
        "No",
        "No"
    ],
    "future_window_used": [
        "No",
        "No",
        "No"
    ]
})

display(feature_audit)

print("\n=== TARGET ===")
print("Target: is_declining")
print("Target window: April 2026")
print("Target uses future outcome: Yes")

print("\n=== IDENTIFIER CHECK ===")
print("client_hash_id used as predictive feature: No")
print("client_hash_id used for grouped validation: Yes")

# Programmatic checks
assert set(feature_cols).issubset({
    "impressions_march",
    "clicks_march",
    "avg_position_march"
})

assert "impressions_april" not in feature_cols
assert "is_declining" not in feature_cols
assert "client_hash_id" not in feature_cols

print("\nLeakage checks passed.")
print("No April outcome field is included in the final predictive features.")

=== FINAL FEATURE SET ===


,feature,source_window,used_for_target,future_window_used
0,impressions_march,March 2026,No,No
1,clicks_march,March 2026,No,No
2,avg_position_march,March 2026,No,No



=== TARGET ===
Target: is_declining
Target window: April 2026
Target uses future outcome: Yes

=== IDENTIFIER CHECK ===
client_hash_id used as predictive feature: No
client_hash_id used for grouped validation: Yes

Leakage checks passed.
No April outcome field is included in the final predictive features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Earlier claim:**

The Logistic Regression model can predict which content pages will experience declining impressions.

**Rewritten claim:**

The Logistic Regression model showed a measured ROC-AUC of 0.538 under client-grouped validation. This provides a limited directional signal for identifying pages associated with subsequent impression decline, but the result is not strong enough to support treating the model as a reliable standalone predictor.

The random-split ROC-AUC of 0.586 was higher than the grouped result, so the grouped evaluation provides the more conservative evidence for generalization to unseen clients.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Claim rewrite check

claim_audit = pd.DataFrame({
    "claim_type": [
        "Earlier claim",
        "Rewritten claim"
    ],
    "claim": [
        "The Logistic Regression model can predict which content pages will experience declining impressions.",
        "The Logistic Regression model provides a limited directional signal for identifying pages associated with subsequent impression decline under client-grouped validation."
    ]
})

display(claim_audit)

print("Evidence used:")
print(f"- Random-split ROC-AUC: {random_auc:.3f}")
print(f"- Client-grouped ROC-AUC: {grouped_auc:.3f}")
print(f"- Difference: {grouped_auc - random_auc:+.3f}")
print("- Leakage audit: passed")
print("- Client overlap in grouped validation: 0")

print("\nClaim language is restricted to observed, measured, directional, and decision-support framing.")

,claim_type,claim
0,Earlier claim,The Logistic Regression model can predict whic...
1,Rewritten claim,The Logistic Regression model provides a limit...


Evidence used:
- Random-split ROC-AUC: 0.588
- Client-grouped ROC-AUC: 0.538
- Difference: -0.050
- Leakage audit: passed
- Client overlap in grouped validation: 0

Claim language is restricted to observed, measured, directional, and decision-support framing.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.